In [13]:
# General
# OPTIONAL: if you want to have more information on what's happening, activate the logger as follows
import os
from pathlib import Path

# Data
import pandas as pd

# Embeddings and ML
import torch
from transformers import BertForSequenceClassification
from word_embeddings import get_bert_word_embeddings

## Create dataframes

In [14]:
from csv import QUOTE_NONE
import sys
import csv
csv.field_size_limit(sys.maxsize)

base_path = Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data"
datasets = {
    "school_shooters": base_path / "school_shooters.csv",
    "manifestos": base_path / "manifestos.csv",
    "stair_twitter_archive": base_path / "stair_twitter_archive.csv",
    "twitter": base_path / "twitter.csv",
}

schoolshootersinfo_df = pd.read_csv(datasets["school_shooters"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
manifesto_df = pd.read_csv(datasets["manifestos"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stair_twitter_archive_df = pd.read_csv(datasets["stair_twitter_archive"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df = pd.read_csv(datasets["twitter"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df

,date,text,name
0,NaN,Dorian Gray with Rainbow Scarf #LoveWins (from...,smile annotations
1,NaN,@SelectShowcase @Tate_StIves ... Replace with ...,smile annotations
2,NaN,@Sofabsports thank you for following me back. ...,smile annotations
3,NaN,@britishmuseum @TudorHistory What a beautiful ...,smile annotations
4,NaN,@NationalGallery @ThePoldarkian I have always ...,smile annotations
...,...,...,...
5052,2016-09-10,"""""""You bet Ben was belting louder than any gir...",umass
5053,2016-09-10,One of my hobby @ Ma Hood https://t.co/SHJDDWQ8QB,umass
5054,2016-09-10,Another Cardigan Records Hopscotch Day Party i...,umass
5055,2016-09-11,Bachelorette 💍💞 @ Laurita Winery https://t.co/...,umass


## Initial testing

One may argue that it could be beneficial to start with a overweigth of regular users as the school shooters posts will by its nature be a fraction. Therefore, I start with the school shooters info dataset (~1000 rows) against twitter (~5000 rows).

#### Text preparation
We create a list of all texts

In [15]:
schoolshooters_texts = schoolshootersinfo_df["text"].to_list()

In [16]:
threat_df = schoolshootersinfo_df.assign(target=1)
no_threat_df = twitter_df.assign(target=0)
combined_df = pd.concat([threat_df, no_threat_df])
combined_df

,date,text,name,target
0,NaN,Oh the happiness I could have had mingling amo...,Seung Hui Cho,1
1,NaN,Only if you could be the victim of your repreh...,Seung Hui Cho,1
2,NaN,"For every action, there is an equal and opposi...",Seung Hui Cho,1
3,NaN,"All the shit you’ve given me, right back at yo...",Seung Hui Cho,1
4,NaN,You had a hundred billion chances and ways to ...,Seung Hui Cho,1
...,...,...,...,...
5052,2016-09-10,"""""""You bet Ben was belting louder than any gir...",umass,0
5053,2016-09-10,One of my hobby @ Ma Hood https://t.co/SHJDDWQ8QB,umass,0
5054,2016-09-10,Another Cardigan Records Hopscotch Day Party i...,umass,0
5055,2016-09-11,Bachelorette 💍💞 @ Laurita Winery https://t.co/...,umass,0


#### Should clean texts before tokenization

In [17]:
import re
def remove_space_between_numbers(text):
    text = re.sub(r'(\d)\s+(\d)', r'\1\2', text)
    return text
def filter_emails(text):
    pattern = r'(?:(?!.*?[.]{2})[a-zA-Z0-9](?:[a-zA-Z0-9.+!%-]{1,64}|)|\"[a-zA-Z0-9.+!% -]{1,64}\")@[a-zA-Z0-9][a-zA-Z0-9.-]+(.[a-z]{2,}|.[0-9]{1,})'
    text = re.sub(pattern, '', text)
    return text
def filter_ref(text):
    pattern = r'(\(*)(ref|réf)(\.|[ ])\d+(\)*)'
    text = re.sub(pattern, '', text)
    return text
def filter_websites(text):
    pattern = r'(http\:\/\/|https\:\/\/)?([a-z0-9][a-z0-9\-]*\.)+[a-z][a-z\-]*'
    text = re.sub(pattern, '', text)
    return text
def filter_phone_numbers(text):
    pattern = r'(?:(?:\+|00)33[\s.-]{0,3}(?:\(0\)[\s.-]{0,3})?|0)[1-9](?:(?:[\s.-]?\d{2}){4}|\d{2}(?:[\s.-]?\d{3}){2})|(\d{2}[ ]\d{2}[ ]\d{3}[ ]\d{3})'
    text = re.sub(pattern, '', text)
    return text

def clean_text(text):
    text = text.lower()
    text = text.replace(u'\xa0', u' ')
    text = filter_phone_numbers(text)
    text = filter_emails(text)
    text = filter_ref(text)
    text = filter_websites(text)
    text = remove_space_between_numbers(text)
    return text

combined_df['cleaned_text'] = combined_df.text.apply(clean_text)

In [18]:
from transformers import DistilBertTokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

encoded_corpus = tokenizer(text=combined_df.cleaned_text.tolist(),
                            add_special_tokens=True,
                            padding='max_length',
                            truncation='longest_first',
                            max_length=300,
                            return_attention_mask=True)

input_ids = encoded_corpus['input_ids']
attention_mask = encoded_corpus['attention_mask']


### Filter out long descriptions

In [19]:
import numpy as np
def filter_long_descriptions(tokenizer, descriptions, max_len):
    indices = []
    lengths = tokenizer(descriptions, padding=False, 
                     truncation=False, return_length=True)['length']
    for i in range(len(descriptions)):
        if lengths[i] <= max_len-2:
            indices.append(i)
    return indices
short_descriptions = filter_long_descriptions(tokenizer, 
                               combined_df.cleaned_text.tolist(), 300)
input_ids = np.array(input_ids)[short_descriptions]
attention_mask = np.array(attention_mask)[short_descriptions]
labels = combined_df.target.to_numpy()[short_descriptions]

Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors


In [21]:
from sklearn.model_selection import train_test_split
test_size = 0.1
seed = 42
train_inputs, test_inputs, train_labels, test_labels = \
            train_test_split(input_ids, labels, test_size=test_size, 
                             random_state=seed)
train_masks, test_masks, _, _ = train_test_split(attention_mask, 
                                        labels, test_size=test_size, 
                                        random_state=seed)

In [22]:
import torch
from torch.utils.data import TensorDataset, DataLoader
batch_size = 32
def create_dataloaders(inputs, masks, labels, batch_size):
    input_tensor = torch.tensor(inputs)
    mask_tensor = torch.tensor(masks)
    labels_tensor = torch.tensor(labels)
    dataset = TensorDataset(input_tensor, mask_tensor, 
                            labels_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, 
                            shuffle=True)
    return dataloader
train_dataloader = create_dataloaders(train_inputs, train_masks, 
                                      train_labels, batch_size)
test_dataloader = create_dataloaders(test_inputs, test_masks, 
                                     test_labels, batch_size)

In [24]:
import torch.nn as nn
from transformers import DistilBertForSequenceClassification, DistilBertModel, DistilBertConfig
class DistilBertForSequenceRegression(nn.Module):
    
    def __init__(self, drop_rate=0.2, freeze=False):
        super(DistilBertForSequenceRegression, self).__init__()
        D_in, D_out = 768, 1    
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.regressor = nn.Sequential(nn.Dropout(drop_rate), nn.Linear(D_in, D_out))
    
    def forward(self, input_ids, attention_masks):        
        outputs = self.bert(input_ids, attention_masks)
        class_label_output = outputs[1]
        outputs = self.regressor(class_label_output)
        return outputs

model = DistilBertForSequenceRegression(drop_rate=0.2)

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_transform.bias']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [25]:
from transformers import AdamW
optimizer = AdamW(model.parameters(),
                  lr=5e-5,
                  eps=1e-8)

In [26]:
from transformers import get_linear_schedule_with_warmup
epochs = 5
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,       
                 num_warmup_steps=0, num_training_steps=total_steps)

In [27]:
loss_function = nn.MSELoss()

In [29]:
from torch.nn.utils.clip_grad import clip_grad_norm
device = "cpu"
def train(model, optimizer, scheduler, loss_function, epochs,       
          train_dataloader, device, clip_value=2):
    for epoch in range(epochs):
        print(epoch)
        print("-----")
        best_loss = 1e10
        model.train()
        for step, batch in enumerate(train_dataloader): 
            print(step)  
            batch_inputs, batch_masks, batch_labels = \
                               tuple(b.to(device) for b in batch)
            model.zero_grad()
            outputs = model(batch_inputs, batch_masks)           
            loss = loss_function(outputs.squeeze(), 
                             batch_labels.squeeze())
            loss.backward()
            clip_grad_norm(model.parameters(), clip_value)
            optimizer.step()
            scheduler.step()
                
    return model
model = train(model, optimizer, scheduler, loss_function, epochs, 
              train_dataloader, device, clip_value=2)

0
-----
0


IndexError: tuple index out of range

In [ ]:
def evaluate(model, loss_function, test_dataloader, device):
    model.eval()
    test_loss, test_r2 = [], []
    for batch in test_dataloader:
        batch_inputs, batch_masks, batch_labels = \
                                 tuple(b.to(device) for b in batch)
        with torch.no_grad():
            outputs = model(batch_inputs, batch_masks)
        loss = loss_function(outputs, batch_labels)
        test_loss.append(loss.item())
        r2 = r2_score(outputs, batch_labels)
        test_r2.append(r2.item())
    return test_loss, test_r2
def r2_score(outputs, labels):
    labels_mean = torch.mean(labels)
    ss_tot = torch.sum((labels - labels_mean) ** 2)
    ss_res = torch.sum((labels - outputs) ** 2)
    r2 = 1 - ss_res / ss_tot
    return r2

In [ ]:
def predict(model, dataloader, device):
    model.eval()
    output = []
    for batch in dataloader:
        batch_inputs, batch_masks, _ = \
                                  tuple(b.to(device) for b in batch)
        with torch.no_grad():
            output += model(batch_inputs, 
                            batch_masks).view(1,-1).tolist()[0]
    return output

In [ ]:
# val_set = val_data[['id_annonce', 'description', 'prix']]
# val_set['cleaned_description'] = \
#                 val_set.description.apply(clean_text)
# encoded_val_corpus = \
#                 tokenizer(text=val_set.cleaned_description.tolist(),
#                           add_special_tokens=True,
#                           padding='max_length',
#                           truncation='longest_first',
#                           max_length=300,
#                           return_attention_mask=True)
# val_input_ids = np.array(encoded_val_corpus['input_ids'])
# val_attention_mask = np.array(encoded_val_corpus['attention_mask'])
# val_labels = val_set.target.to_numpy()
# val_dataloader = create_dataloaders(val_input_ids, 
#                          val_attention_mask, val_labels, batch_size)
# y_pred_scaled = predict(model, val_dataloader, device)